# CICIDS-2017 CCS Benchmark
**Runtime**: GPU (T4). Runtime → Change runtime type → T4 GPU before running.

**Data**: downloaded automatically from Kaggle (`chethuhn/network-intrusion-dataset`).  
Enter your Kaggle credentials in the cell below — no Google Drive needed.

In [ ]:
# ── Kaggle credentials — replace with your actual API token ──────────────────
import os
os.environ["KAGGLE_USERNAME"] = "your_username"
os.environ["KAGGLE_KEY"]      = "your_key"

# ── Benchmark config ──────────────────────────────────────────────────────────
MAX_SAMPLES = 500_000   # 0 = full dataset; 500k is safe on free Colab (~12 GB RAM)
SEEDS       = 3
NO_XGB      = False
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
# Download CICIDS-2017 from Kaggle
!pip install -q kagglehub

import kagglehub
DATA_PATH = kagglehub.dataset_download("chethuhn/network-intrusion-dataset")
print("Dataset path:", DATA_PATH)

In [ ]:
!pip install -q torch scikit-learn pandas pyarrow xgboost

In [ ]:
# Clone the repo
import os
if not os.path.exists('/content/netadv-ccs'):
    !git clone https://github.com/sassom2112/catt-ccs /content/netadv-ccs
os.chdir('/content/netadv-ccs')
!git pull

In [ ]:
import sys, pathlib
sys.path.insert(0, '/content/netadv-ccs')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from netadv.attacks.fgsm import fgsm
from netadv.attacks.pgd import pgd
from netadv.attacks.transfer import transfer_table
from netadv.classifiers.sklearn_wrap import train_random_forest, train_xgboost
from netadv.constraints.bounds import ConstraintBounds, validity_report
from netadv.constraints.datasets.cicids2017 import CICIDS2017_SPEC, NUM_FEATURES
from netadv.eval.calibration import calibration_table, CICIDS_REPRESENTATIVE
from netadv.eval.metrics import evasion_rate

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

LABEL_COL      = 'Label'
ATTACK_CAT_COL = 'attack_cat'
EPSILONS       = [0.05, 0.10, 0.20, 0.30, 0.40, 0.50]

In [ ]:
def _load_data(data_path):
    p = pathlib.Path(data_path)
    if p.suffix == '.parquet':
        return pd.read_parquet(p)
    if p.is_dir():
        csvs = sorted(p.glob('*.csv'))
        print(f'  Found {len(csvs)} CSV(s): {[f.name for f in csvs]}')
        df = pd.concat([pd.read_csv(f, low_memory=False) for f in csvs], ignore_index=True)
    else:
        df = pd.read_csv(p, low_memory=False)
    df.columns = df.columns.str.strip()
    df.replace([float('inf'), float('-inf')], float('nan'), inplace=True)
    for col in ('Init_Win_bytes_forward', 'Init_Win_bytes_backward'):
        if col in df.columns:
            df[col] = df[col].replace(-1, 0)
    nonneg = [f for f, b in CICIDS2017_SPEC.bounds.items() if b.lo == 0.0 and f in df.columns]
    df[nonneg] = df[nonneg].clip(lower=0)
    if LABEL_COL in df.columns:
        df[ATTACK_CAT_COL] = df[LABEL_COL].str.strip()
        df[LABEL_COL] = (df[ATTACK_CAT_COL].str.upper() != 'BENIGN').astype(int)
    return df

print(f'Loading {DATA_PATH}…')
df = _load_data(DATA_PATH)
if MAX_SAMPLES and MAX_SAMPLES < len(df):
    df = df.sample(MAX_SAMPLES, random_state=0)
    print(f'Subsampled to {MAX_SAMPLES:,} rows')
print(f'Using {len(df):,} rows  |  labels: {dict(df[LABEL_COL].value_counts())}')

In [ ]:
class _MLP(nn.Module):
    def __init__(self, input_dim, hidden=(256, 128, 64), dropout=0.3):
        super().__init__()
        layers, prev = [], input_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x).squeeze(-1)

def _train_mlp(X_tr, y_tr, X_val, y_val, epochs=30, patience=5):
    from torch.utils.data import DataLoader, TensorDataset
    n_pos = (y_tr == 1).sum(); n_neg = (y_tr == 0).sum()
    pw = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32).to(device)
    crit = nn.BCEWithLogitsLoss(pos_weight=pw)
    loader = DataLoader(
        TensorDataset(torch.tensor(X_tr, dtype=torch.float32),
                      torch.tensor(y_tr, dtype=torch.float32)),
        batch_size=2048, shuffle=True)
    Xv = torch.tensor(X_val, dtype=torch.float32).to(device)
    model = _MLP(X_tr.shape[1]).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=3, factor=0.5)
    best_f1, best_state, no_imp = 0.0, None, 0
    for ep in range(1, epochs + 1):
        model.train(); total = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(); loss = crit(model(xb), yb)
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); total += loss.item() * len(xb)
        model.eval()
        with torch.no_grad():
            preds = (model(Xv).cpu().numpy() > 0).astype(int)
        vf1 = f1_score(y_val, preds, zero_division=0)
        sched.step(total / len(X_tr))
        print(f'  epoch {ep:3d}  loss={total/len(X_tr):.4f}  val_f1={vf1:.4f}')
        if vf1 > best_f1:
            best_f1, best_state, no_imp = vf1, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            no_imp += 1
            if no_imp >= patience:
                print(f'  early stop (best val_f1={best_f1:.4f})'); break
    model.load_state_dict(best_state); model.eval()
    return model

def _build_pipeline(X_raw):
    pre = ColumnTransformer([('num', Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value=0.0)),
        ('scaler', StandardScaler())]), NUM_FEATURES)])
    p = Pipeline([('prep', pre)]); p.fit(X_raw); return p

In [ ]:
def run_seed(df, seed):
    X_raw = df[NUM_FEATURES]
    y = df[LABEL_COL].to_numpy(dtype=int)
    X_tr_raw, X_te_raw, y_tr, y_te = train_test_split(
        X_raw, y, test_size=0.2, random_state=seed, stratify=y)
    pipeline = _build_pipeline(X_tr_raw)
    X_tr = pipeline.transform(X_tr_raw).astype(np.float32)
    X_te = pipeline.transform(X_te_raw).astype(np.float32)
    bounds = ConstraintBounds.from_spec(CICIDS2017_SPEC, pipeline)
    no_bounds = ConstraintBounds(
        lb=np.full(bounds.n_total, -np.inf, dtype=np.float32),
        ub=np.full(bounds.n_total,  np.inf, dtype=np.float32),
        n_num=bounds.n_num, n_total=bounds.n_total)
    report = validity_report(X_te, bounds)
    print('✓ validity_report: zero violations' if report.empty else f'WARNING: {len(report)} violations')
    print(f'  [seed={seed}] Training MLP…')
    X_tr2, X_val, y_tr2, y_val = train_test_split(X_tr, y_tr, test_size=0.15, random_state=seed, stratify=y_tr)
    mlp = _train_mlp(X_tr2, y_tr2, X_val, y_val)
    print(f'  [seed={seed}] Training RandomForest…')
    rf = train_random_forest(X_tr, y_tr, seed=seed)
    targets = [rf]
    if not NO_XGB:
        try:
            print(f'  [seed={seed}] Training XGBoost…')
            targets.append(train_xgboost(X_tr, y_tr, seed=seed))
        except ImportError as e:
            print(f'  XGBoost skipped: {e}')
    with torch.no_grad():
        cp = (mlp(torch.tensor(X_te).to(device)).cpu().numpy() > 0).astype(int)
    print(f'  [seed={seed}] MLP  F1={f1_score(y_te, cp, zero_division=0):.4f}  acc={(cp==y_te).mean():.4f}')
    for t in targets:
        tp = t.predict(X_te)
        print(f'  [seed={seed}] {t.name:14s}  F1={f1_score(y_te, tp, zero_division=0):.4f}  acc={(tp==y_te).mean():.4f}')
    rows, transfer_rows = [], []
    for eps in EPSILONS:
        alpha = eps / 40 * 2.5
        for label, b in [('constrained', bounds), ('unconstrained', no_bounds)]:
            xf = fgsm(mlp, X_te, y_te, epsilon=eps, bounds=b, device=device)
            xp = pgd(mlp, X_te, y_te, epsilon=eps, alpha=alpha, n_steps=40, bounds=b, device=device)
            rows.append({'seed': seed, 'dataset': 'CICIDS-2017', 'epsilon': eps, 'variant': label,
                         'classifier': 'MLP',
                         'fgsm_evasion': evasion_rate(mlp, xf, y_te, device=device),
                         'pgd_evasion':  evasion_rate(mlp, xp, y_te, device=device)})
            if targets:
                for r in transfer_table(X_te, y_te, xp, xp, targets):
                    if r['variant'] == 'constrained':
                        transfer_rows.append({'seed': seed, 'epsilon': eps,
                                              'attack_variant': label,
                                              'target': r['classifier'],
                                              'transfer_evasion': r['evasion_rate']})
        print(f'  [seed={seed}] ε={eps:.2f} done')
    return rows, transfer_rows, pipeline

all_rows, all_transfer, last_pipeline = [], [], None
for seed in range(SEEDS):
    rows, tr, pipeline = run_seed(df, seed)
    all_rows.extend(rows); all_transfer.extend(tr); last_pipeline = pipeline

In [ ]:
results_df = pd.DataFrame(all_rows)
agg = (results_df.groupby(['dataset', 'epsilon', 'variant', 'classifier'])
       [['fgsm_evasion', 'pgd_evasion']].agg(['mean', 'std']).reset_index())
agg.columns = ['_'.join(c).strip('_') for c in agg.columns]
print('── White-box MLP evasion ─────────────────────────────────────────────────')
display(agg[agg['classifier'] == 'MLP'])

if all_transfer:
    tr_df = pd.DataFrame(all_transfer)
    tr_agg = (tr_df.groupby(['epsilon', 'attack_variant', 'target'])
              ['transfer_evasion'].agg(['mean', 'std']).reset_index())
    print('── Transfer evasion ──────────────────────────────────────────────────────')
    display(tr_agg)
    tr_agg.to_csv('results_transfer_cicids.csv', index=False)

if last_pipeline:
    cal = calibration_table(last_pipeline, feature_names=NUM_FEATURES,
                            epsilons=[0.10, 0.20, 0.40],
                            representative_features=CICIDS_REPRESENTATIVE)
    print('── ε calibration ─────────────────────────────────────────────────────────')
    display(cal)
    cal.to_csv('results_calibration_cicids.csv', index=False)

results_df.to_csv('results_ccs_cicids.csv', index=False)
print('\nSaved: results_ccs_cicids.csv')

In [ ]:
from google.colab import files
for f in ['results_ccs_cicids.csv', 'results_transfer_cicids.csv', 'results_calibration_cicids.csv']:
    try: files.download(f)
    except Exception: pass